# Data Exploration with DataFrames (pandas)

# Data exploration with DataFrames, cont.

In [ ]:
#@title
# make sure to run this cell to import the external files we need for today
# and load in the appropriate packages
!git clone https://github.com/ccbskillssem/pythonbootcamp.git

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

alleles = pd.read_table('/content/pythonbootcamp/day_4/alleles.tsv')
clinvar = pd.read_table('/content/pythonbootcamp/day_4/clinvar.tsv')

## Changing labels

### Row labels

Currently, the row labels for `alleles` are numeric, which makes it easy to index rows just as we did with arrays. However, it's also possible to use row *names* as labels, just as we do with columns.

To do this, we can select an existing column of the DataFrame and turn it into the row Index. However, it's **extremely** important that we make sure the candidate column contains only unique values: we wouldn't want to try and index a row name, only to retrieve multiple rows with the same name!

Series have a useful attribute called `.is_unique`, which is `True` if all values in the Series are unique. Given that columns of DataFrames are stored as Series, we can easily check our candidate columns.

In [ ]:
# try it out:
# print the .is_unique attribute of each column
# hint: remember for loops? try using a loop with the column labels
print(alleles['chromosome'].is_unique)
print(alleles['position'].is_unique)

In [ ]:
for col in alleles.columns:
  print(col + "", alleles[col].is_unique)

Great – looks like there's a candidate column for our row labels. We can now change `allele`'s row labels using the `.set_index()` method.

In [ ]:
# in this case, we're opting to update the variable

alleles = alleles.set_index('ID')
alleles.head()

Excellent! As you can see, each row's `ID` value is now the row label. We can access each row using `.loc[]` and the corresponding `ID` label.

In [ ]:
alleles.index

In [ ]:
alleles.loc['rs573303859']

Row names can be convenient, but also cumbersome: for example, now that we've assigned `ID` as our row labels, all of the row labels are strings: thus, we can no longer slice multiple rows with `.loc[]`.

> If you really want to slice multiple rows *and* retain string-type row labels, you will need to use `.iloc[]` instead of `.loc[]`. `.iloc[]` is used to access columns/rows/slices/cells in a `numpy` array-like manner. You can read more about it [here](https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#selection-by-position).

In [ ]:
# try it out:
# access the row labeled rs563299082
# and its chromosome, position, reference, and alternative values
alleles.loc['rs563299082']

In [ ]:
alleles.loc['rs563299082',['chromosome', 'position', 'reference', 'alternative']]

If you end up wanting to revert back to the standard 0-index numeric row labels, you can use the `.reset_index()` method to convert the row names back to a regular column.

In [ ]:
# let's return alleles to its original state

alleles = alleles.reset_index()
alleles.head()

### Column labels
In the case that we want to replace column labels, we can use the `.rename()` method, which is the equivalent of `.replace()` for column labels. Just as we showed you before, the easiest way to do this is to provide a dictionary of target and substitute labels.

For example, let's consider `reference` and `alternative`, which are commonly shortened to `REF` and `ALT` in genomic file formats. Changing the column labels while maintaining the column content is a single-line operation:

In [ ]:
# rename reference and alternative to REF and ALT for all rows

alleles.rename({'reference': 'REF',
                 'alternative': 'ALT'}, axis = 'columns').head()

## Sorting by column(s)
There's no way to sugar-coat it: there was no intuitive way for us to sort arrays by specific columns in `numpy`.

With `pandas`, the `.sort_values()` method allows us to sort a DataFrame/Series with the following inputs:
* A single column name, or a list of column names
* The `ascending` parameter, which takes a single Boolean value or a list of Boolean values (corresponding to the list of column names).
  * `True` will sort by lowest to highest (ascending).
  * `False` will sort by highest to lowest (descending).

In [ ]:
# let's sort AFR_AF by highest to lowest frequency

alleles.sort_values("AFR_AF", ascending=False).head()

⏸ **Exercise 1** Try sorting `alleles` by *all three* of the following columns:
* `AFR_AF`, from highest to lowest.
* `AMR_AF`, from lowest to highest.
* `EAS_AF`, from highest to lowest.

In [ ]:
### write your code below ###
alleles.sort_values("AFR_AF", ascending=False).head()

## Adding columns

We can easily create new columns by simply assigning a new column name and contents to a DataFrame.

In [ ]:
# let's sample some rows with .sample()
# using 2024 as our seed value

some_alleles = alleles.sample(n = 5, random_state = 2024)
some_alleles

In [ ]:
some_alleles['sample_col'] = 0 # a new column full of zeros
some_alleles['exp'] = "exp_1"
some_alleles

---
⏸ **Exercise 2 (No coding)** What happens when you change the seed value of a random sampler (like `.sample()`)? What happens if you don't have a seed value specified?

Hint: One of the morning exercises might help you answer this question.

---

Next, we can also assign new columns by explicitly providing a column label and a "definition" for the column.

In [ ]:
# let's reuse some_alleles
# creating a new column that contains the *reference* allele frequency
# this takes advantage of numpy broadcasting

some_alleles['reference_AF'] = 1 - some_alleles['AF']
some_alleles

Lastly, the `.map()` method allows us to **mutate** new columns based on values of existing columns. This is useful for adding categorical or Boolean columns without overwriting existing columns.

In [ ]:
alleles['filter'].head()

In [ ]:
# let's mutate a new column called filter_pass
# instead of replacing the values of filter

alleles['filter_pass'] = alleles['filter'].map({'PASS': True,
                                      'LowQ': False})
alleles.head()

## Dropping columns

We can use the `.drop()` method to drop columns as easily as we created them. This can be handy if you end up with too many columns during your exploration, or if you simply don't need some of the columns that came with your dataset.

In [ ]:
# getting rid of filter_pass
# removing it from all rows in the dataframe

alleles = alleles.drop('filter_pass', axis='columns')
alleles

## Grouping

We've now reached the operations that are *only* possible in `pandas`. **Grouped operations** partition the dataset into subsets based on the value of a given column or columns prior to operation.

There are two major steps in performing grouped operations:
1. **Splitting**: Splitting data into groups.
2. **Applying**: Applying a function across separate groups. (This is where vectorization is helpful!)

We can accomplish the first step using the `.group_by()` method, which takes one or more column labels.

In [ ]:
alleles[alleles["filter"] == "LowQ"]
alleles[alleles["filter"] == "PASS"]

In [ ]:
# this method doesn't show a visible output, it creates a groupby object

alleles.groupby('filter')

In [ ]:
alleles[alleles.filter] == "PASS"

In order to see the effect of splitting with `.groupby()`, we need to chain `.groupby()` with a function that we want applied to our groups.

For example, let's say that we want to know the average quality score of alleles that pass (or don't pass) filter.

In [ ]:
alleles.groupby('filter')['quality'].mean()

## Merging (review)

This morning we discussed merging two datasets: `alleles` and `clinvar`. Merging is a very useful operation that we'll be using in tomorrow's final mini-project.

Once again, merge `alleles` and `clinvar` so that we have a single DataFrame called `merged`, which contains only the alleles present in both of the datsets.

In [ ]:
# try it out:
# merge alleles and clinvar as you did earlier today into a 'merged' dataframe: inner join on the ID columns
merged_df = pd.merge(alleles, clinvar, 'inner', on='ID')

In [ ]:
merged_df

In [ ]:
merged_df['chromosome_x'] == merged_df['chromosome_y']

In [ ]:
(merged_df['chromosome_x'] == merged_df['chromosome_y']).all()

## Exporting

This is our final lesson in `pandas`: hooray!

We've learned a lot about how to parse and manipulate our DataFrames. At some point in the near future, you'll likely want to save the results of your operations.

The `.to_csv()` method allows us to easily export our DataFrame to a `.csv` file. As the name implies, it will save a DataFrame to a `.csv` file, given a string (file name).

Let's go ahead and cap off our lecture by exporting `merged` to a file called `merged.csv`.

In [ ]:
merged_df.to_csv('/content/pythonbootcamp/day_5/my_first_merged.csv')

In [ ]:
# Try it out
# Read back in that same file to see how it was saved
reload_merged_df = pd.read_csv('/content/pythonbootcamp/day_5/my_first_merged.csv')
reload_merged_df

If you open the `Files` menu (left panel, folder icon), you'll see that the `merged.csv` file is now available.
___
**CAUTION**: Files that you save while using Colab are not retained after you close the notebook, as they only exist in Colab's temporary **session storage**. If you generate files and wish to keep them, make sure to download your files (with the same three dots menu) before you exit Colab.
___

That's it for our introduction to `pandas`! We hope you're starting to get a feel for how immensely powerful `pandas` can be. (Perhaps too powerful for a single bootcamp lecture...)

* To learn more about DataFrame methods, click here for the DataFrame documentation.
* To learn more about Series methods, click here for the DataFrame documentation.
* To learn about other common `pandas` routines, refer to the official `pandas` cookbook [here](https://pandas.pydata.org/docs/user_guide/cookbook.html). (No pandas were harmed in the making of this cookbook.)

# << `Exercises` >>

⏸ **Exercise 3a**: Just like with arrays, we can use the `.isna()` method to detect missing values in a DataFrame. Use this to identify all missing values in `clinvar`.

In [ ]:
### write your code below ###
char.isna()

⏸ **Exercise 3b** Let's chain together some `pandas` methods! Which columns of `clinvar` are missing values?

*Hint*: The `.any()` method returns a `True` if any of the values in the Series or array are `True`.

In [ ]:
### write your code below ###
clinvar.isna().any(axis=0)

⏸ **Exercise 3c**: Find the total number of missing values in each column of `clinvar`.

*Hint*: Recall that `False` Booleans are considered equivalent to `0`, and `True` values are considered equivalent to `1`.

In [ ]:
### write your code below ###
clinvar.isna().sum(axis=0)

⏸ **Exercise 3d**: Which rows in `clinvar` are incomplete? Filter `clinvar` to show these rows.

*Hint*: You can change the axis along which `.any()` operates.

In [ ]:
### write your code below ###
filter_mask_with_na = (clinvar.isna().any(axis=1) == True)
clinvar[filter_mask_with_na]

In [ ]:
clinvar[clinvar.isna().any(axis=1)]

⏸ **Exercise 4a** We're now going to move to data cleaning.

First, select rows in ```clinvar``` that have complete data (no missing values): save this filtered DataFrame to a variable called `clinvar_complete`. What are the dimensions of `clinvar_complete`?

In [ ]:
### write your code below ###
clinvar.isna()

In [ ]:
print("per row:", clinvar.isna().sum(axis=1))

⏸ **Exercise 4b**: It seems like a significant number of `clinvar` rows contain missing values. Calculate the mean, median, and maximum number of missing values for `clinvar` rows.

*Hint*: This exercise uses the same principle as Exercise 3c.

In [ ]:
clinvar.head()

In [ ]:
### write your code below ###
clinvar.isna().sum(axis=1).mean()

In [ ]:
clinvar.isna().sum(axis=1).max()

In [ ]:
clinvar.isna().sum(axis=1).median()

In [ ]:
clinvar_partial = clinvar.diploma()
clinvar_partial.isna().sum(axis=1).max()

⏸ **Exercise 4c**: The `.dropna()` method accomplishes what we did with Boolean filtering, but it allows for additional flexibility when filtering rows. Look up the documentation for `.dropna()`, then use it to select rows in `clinvar` that have no more than 2 NA values. Assign this DataFrame to `clinvar_partial`.

In [ ]:
alleles = alleles.drop('filter', axis='columns')
alleles

In [ ]:
### write your code below ###
alleles.dropna()

⏸ **Exercise 4d**: How many rows remain compared to `clinvar_complete` and the original `clinvar` DataFrame?

In [ ]:
### write your code below ###
print("original data dimensions:", clinvar.shape)
print("complete data dimensions:", clinvar_complete.shape)
print("partial data dimensions:", clinvar_partial.shape)

⏸ **Challenge**: Create a new DataFrame called `pathogenic` that contains only pathogenic variants.

*Hint*: The `clinical_significance` column contains several different strings with the substring `'pathogenic'`. How can you take advantage of this to filter `clinvar`? If you're not sure, check the morning notebook.)

In [ ]:
### write your code below ###
clinvar['clinical_significance'].unique()

Find the number of pathogenic variants for each of three genes: `BRCA1`, `PAH`, and `CFTR`.

In [ ]:
### write your code below ###
pathogenic[pathogenic['gene'].isin(['BRCA1','PAH','CFTR'])].value_counts('gene')

In [ ]:
count = pathogenic['gene'].value_counts()
mask = count.index.isin(['BPCA1', 'CFTR', 'PAN'])

---
# Introduction to Machine Learning

# Introduction to Machine Learning

*This notebook was written by Carmelle Catamura and Prakruthi Burra for the CCB Python Bootcamp*

## What is Machine Learning?

Machine learning is the science of  programming computers so they can learn from data. In other words, the computer tries to learn *patterns* (that are usually hidden or not easily observable) from huge amounts of data during *training* so that it can make accurate predictions in new, unseen data.

## *Why* learn ML?

Machine Learning (ML) is all around us. From self-driving cars to your search engine results, ML is integrated in our everyday lives. In biology, ML is  used in various research fields from predicting protein structure (think AlphaFold!) to evaluating disease risk scores. Certainly, ML is a very powerful tool that can help accelerate your research!

**It will take more than a day (and more than a notebook!) to master machine learning and understand how to use these tools effectively and responsibly. It requires time to build foundational knowledge, to learn how to implement ML methods, and to grasp key concepts thoroughly. The field is also evolving rapidly and keeping up takes time and work.**

Consider this notebook a quick dip into the ML space. We aim to give you a simple introduction to key machine learning concepts and techniques :)  Moving forward, we strongly encourage you to check out other online resources that are linked at the end of this section.

In this notebook, we are going to tackle the [Wisconsin Breast Cancer Dataset (WBCD)](https://archive.ics.uci.edu/dataset/15/breast+cancer+wisconsin+original) with both classical and modern machine learning approaches, but more on that later..

## Key ML terms

* **Model:** a mathematical framework or system designed to process data and generate predictions. There are different types of models, tailored to specific tasks or data structures.

* **Train/Training:** the process of feeding data into a machine learning algorithm to help the model learn patterns. During training, the model adjusts its internal parameters to minimize the error or loss between its predictions and the real (target) values.

* **Test/Testing:** evaluating a trained model's performance on a separate subset of the data, called the test set, that was not used during training. The goal is to measure how well the model generalizes what it has learned to new, unseen data.

* **Accuracy**: a  metric that measures the model's performance. It's the percentage of correct predictions made by a model out of all the predictions. Depending on the problem you are solving, there are different ways to measure accuracy. In classification tasks, like the one we are doing today, it’s calculated as following -

$$ Accuracy = \frac{\# Correct Predictions}{\# Total Predictions} $$

* **Validate/Validation:** the process of fine-tuning and assessing a model during its training phase using a separate subset of the data, called the validation set. It helps in selecting the best model parameters by preventing overfitting. We will talk more about overfitting later.

* **Task**: refers to the specific problem the model is designed to solve. Common tasks include classification (e.g., species identification), regression (e.g., predicting gene expression levels), clustering (e.g., grouping similar cell types). In this notebook, we will be implementing a classification task on our breast cancer tumor samples.

    **Classification**: Classification involves predicting whether an input sample belongs to one of a few categories

    **Regression**: Regression involves predicting a continuous output from the features of your inputs

## Unsupervised and Supervised Learning

Machine learning systems can often be classified according to what supervision they get during training. For example,

**Supervised learning** refers to when the training set being fed to the algorithm includes the correct answers, which can either be categorical or continuous values.

**Unsupervised learning** refers to when the training data is unlabeled and the system tries to learn the underlying structure in the data.

Today, we will be implementing supervised learning to train different models with our breast cancer tumor data.

## The ML Paradigm

Here is a conceptual framework for machine learning. This framework is useful because it allows you to check if each of the pieces of the machine learning  approach you have makes sense.
* **Data:** We have data that we use to train a model and data that we use to test the model's performance on. \\
* **Model:** Each model has a particular form. Within that form, there is flexibility. This will become clear to us as we look at concrete examples. \\
* **Loss:** These models are trained to minimize a loss or an error. This loss or error is sometimes also referred to as an objective function. \\
* **Optimization:** The minimization procedure is carried out by an optimization algorithm. \\

In this notebook, we will explore a few key ideas regarding

*   Data
*   Forms of some machine learning models

The loss or error function is a key piece of the ML paradigm that we won't get into. But, if we had to recommend the first few loss functions that you should look at - the mean square error and cross-entropy Loss would be the top picks. We will also not get into the optimization algorithms that are used in machine learning. There are entire textbooks on, and classes in Optimization that can help you work through these.

## Data Exploration

The key to implementing machine learning techniques well lies in how much we know about our data!

The Wisconsin Cancer Dataset is a popular dataset that is often used as a dataset to understand and demonstrate ML classification techniques.
Our goal is classify each tumor sample/datapoint as benign (non-cancerous) or malignant (cancerous) based on various features derived from a digital image of a fine-needle aspirate (FNA) of a breast mass.

This datasest can be accessed via the UCI Machine Learning Repository or via ```sklearn``` built-in datasets.

**First, let's set up our questions: What do we want to learn from our data? What do we plan to predict with our model?** \\
Maybe we want to look at our data and see if there are features that separate the two kinds of tumors? Maybe we want to see if there are correlated features in our dataset? \\
We can theoretically predict many things from any given dataset. But, of course, we want to find something interesting and reasonable to predict from our data. In this notebook, we will try predicting whether a particular tumor is benign or malignant!

In [ ]:
# We are going to be using a Python module called scikit-learn for our tasks
# scikit-learn is a very useful package for ML.
# We will import scikit-learn by running the import statement.
# Note: in our import statement, we have sklearn and not scikit-learn!
import sklearn

Scikit-learn is an easy-to-use package that includes a lot of useful tools for machine learning including dataprocessing, feature selection, hyperparameter tuning, etc. It also includes pre-written models that should cover most ML tasks! Learn more about ```scikit-learn``` [here](https://scikit-learn.org/stable/).

In [ ]:
# Importing the dataset
# There are several popular datasets that are easily accessible to us through packages
# sklearn allows us to import the breast cancer dataset easily
from sklearn.datasets import load_breast_cancer

In [ ]:
# Let's load the dataset
data = load_breast_cancer()
data

In [ ]:
# Let's reorganize our data to something we're more familiar with -- a pandas dataframe that we can parse easily!

# In any sort of data analysis task, numpy is very useful!
# We'll import numpy along with pandas

import numpy as np
import pandas as pd

cancer_data = pd.DataFrame(data=data.data, columns=data.feature_names)
cancer_data.head()

Here, each datapoint in our dataframe is a tumor. The different columns are the "features" that describe each datapoint. These attributes or characteristics of the data are used to make predictions.

In [ ]:
# TO DO
# Can you print the shape of the dataframe?
# How many datapoints do we have? How many features do we have?
cancer_data.shape

# You should have 569 individual datapoints and 30 features per datapoint

In [ ]:
# Let's take a quick peek at our features/columns

cancer_data.columns

We want to use these features to predict if the tumor is malignant (0) or benign (1). These target values are not in the dataframe yet, so let's add them.

In [ ]:
# sci-kit stores our dataset in data.data and the target values in data.target

# TO DO: add the data.target values to our cancer_data dataframe and print the dataframe!
# We want to have the data.target values stored in the column with name 'target' in the cancer_data dataframe
# Hint: df['new_col'] = new_col_values

cancer_data['target'] = data.target #malignant and benign

In [ ]:
# TO DO: Let's inspect our dataset, how many benign (1) and malignant (0) tumors do we have in our dataset?
# Think value_counts()

cancer_data['target'].value_counts()
# 357 datapoints are benign (1) and 212 are malignant (0)

The "target" column now holds the truth values for our classification task. We will train a model on a subset of our data (train set) and then use it predict whether a tumor is malignant (0) or benign (1) on a separate subset of the data (test set).

Because we are going to implement a **supervised learning** approach, the model will see the "target" values during training. We will then evaluate our model's performance based on how well it predicts the "target" values for each point in the test set.

Now that we've set up the problem, let's first do some Exploratory Data Analysis (EDA)! EDA is a crucial step in the machine learning process because it helps us to understand the data, uncover covariates, identify outliers. All of this can inform our decision on how to process the data before using it to train our model.

There's a common saying in ML: *\"Garbage in, garbage out."*  This points to the fact that a model trained on bad data will only produce bad results. Our models can only be as good as our data so we have to make sure that we are controlling for high-quality inputs and pre-processing them appropriately. EDA is a very important step in getting to know your data!

In [ ]:
# What are the features that describe the data?
# We will use df.columns and see if we have the 'target' column in the list of columns
# Everything except the column with column name 'target' is a feature

cancer_data.columns

In [ ]:
# TO DO: check if the dataframe has nan values, if there are replace those nan values with 0!
# Hint: First, check if there are any nan values in the dataframe
# Your code goes here
cancer_data.isna().sum()

In [ ]:
# TO DO: Inspect the datatypes of our features/columns
# Your code goes here
cancer_data.columns

In [ ]:
# TO DO:
# Let's look at some summary statistics using .describe()
# Your code goes here
cancer_data.describe()

A *correlation* describes a statistical relationship between features. It could be that one feature is highly dependent on the other and changing one can drastically affect other features. Let's look into possible highly correlated features in our data.

In [ ]:
import matplotlib.pyplot as plt

# we will be using a cool plotting library called seaborn!
import seaborn as sns

plt.figure(figsize=(14,14))

# we will do this using .corr()
correlation = cancer_data.corr(method='pearson') #alternatively, you can use 'spearman'

# take a look at the correlation variable by printing it in another cell and printing it if you'd like

sns.heatmap(correlation,
           xticklabels=correlation.columns.values,
           yticklabels=correlation.columns.values,
           annot = False, # put True if you want it to show the correlation degree on cell
           fmt='0.1g', # if annot=True, how many significant to display
           cmap= 'coolwarm',
           linewidths=3,
           linecolor='white',
           square=True,
           cbar_kws= {'orientation': 'vertical'},
           annot_kws = {'size':8}
           )

plt.title("Pearson Correlation ")

plt.show()

The bright red blocks outside of the diagonal are high positively correlated features while the bright blue blocks are high negatively correlated features. Do any of these correlations seem reasonable to you?

If we know the radius of something, do we have a decent approximation of its area and perimeter?

If we keep the mean radius and worst radius, can get rid of mean perimeter and mean area, and worst perimeter and worst area?

We can use the results of our correlation analysis to drop some features. What are the benefits of doing this?

1. We reduce redundancy by removing features that provide the same information.
2. Removing some features will make interpretation of the results of our model much simpler.
3. For a smaller dataset like WBCD it probably won't matter much, but when you're training on hundreds of thousands of data, reducing the number of features will increase computational efficiency.

In [ ]:
cancer_data.columns

*Feature engineering*, specifically *feature selection*, is the process of choosing or designing the most important features to use in a machine learning model.

We have to be intentional about what features to train the model with. But, of course, we can't just throw away  features randomly. We must make sure the model's performance won't suffer for it! How are we going to do that?

Let's remind ourselves of our goal here. The task is to predict whether a tumor is malignant or benign. We've done a correlation analysis to see if there are features that we may want to drop. We can also look at the distribution of features for the two types of tumors.

In [ ]:
num_features = len(cancer_data.columns)-1 # exclude target column
n_cols = 3  # Number of columns in the grid
n_rows = (num_features+n_cols-1) // n_cols  # Calculate the number of rows

# Create the subplots
fig, axes = plt.subplots(n_rows, n_cols, figsize=(8, n_rows * 2), constrained_layout=True)
axes = axes.flatten()  # Flatten the axes array for easy iteration

# Plot each feature as a histogram
for i, feature in enumerate(cancer_data.columns[:-1]): #exclude target column
    sns.kdeplot(
        data=cancer_data,
        x=feature,
        hue='target',
        ax=axes[i],
        palette={0: 'magenta', 1: 'blue'},
        legend=True,
    )
    axes[i].set_title(feature)
    axes[i].set_ylabel('Density')

plt.show()

What can you observe? For which features do the distributions of benign and malignant tumor cells look distinct? For example look at "worst concave points" and see how distinct the two distributions are compared to, say, "smoothness error."

In a classification task like this, when the two target classes (in this case, 1 - benign and 0 - malignant) are nicely partitioned, it means that the feature is a good predictor of the target class. Of course, we can't just use the "worst concave points" feature as our only predictor, we have to use it in combination with other strong predictors to ensure that we can get the best accuracy possible.

In [ ]:
# Let's keep our best features and remove others
drop_features = ['mean area', 'mean perimeter', 'worst radius', 'worst area', 'worst perimeter', 'worst texture', 'mean concavity', 'perimeter error', 'area error']
cancer_data_filtered  = cancer_data.drop(columns=drop_features)
cancer_data_filtered

Now that we've selected our best features to train the model, let's work on actually training it!

##Train and test split

First, let us delineate our inputs and outputs. Generally, inputs are denoted by the variable X and outputs by the variable y.

In [ ]:
# From the cancer data, what are columnns that we want to have in our inputs?
# Can we create our input by just dropping the columns that we don't want to keep?
X = cancer_data_filtered.drop(['target'], axis=1)
y = cancer_data_filtered['target']

In [ ]:
# TO DO: print the dimensions of X and y
# Let's check that the input and output shapes make sense to us.
# Your code goes here
print(X.shape, y.shape)

#split into train and test

We will now split our data into train dataset and test dataset. The train dataset will be used to train our models. We will evaluate how well our model is performing on the test dataset. A common split of train and test data is 80:20 i.e. 80% of all the data you have is used to train the model and 20% is used to test the model. That is what we will be doing for the next few examples.

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=41)

In [ ]:
# TODO:
# Print the sizes of X_train, X_test, y_train, y_test. What do you expect to see?
# Your code goes here!
print(X_train.shape, X_test.shape, y_train.shape, y_test.shape)

##Classical Approaches to ML

###Logistic Regression

Logistic regression is a fundamental ML classification method. \\
Here's a brief explanation of what this method does - it computes a linear combination of the different features, let's call this linear combination *l*. It runs this linear combination through a sigmoid function to obtain a probability that the output belongs to **class 1**. \\

In [ ]:
# We will now plot the sigmoid function!
import numpy as np
import matplotlib.pyplot as plt

def sigmoid(x):
  return 1 / (1 + np.exp(-x))

x = np.linspace(-10, 10, 100)
s = sigmoid(x)

plt.plot(x, s)
plt.xlabel("x")
plt.ylabel("Sigmoid(x)")
plt.title("Sigmoid Function")
plt.show()

In our case, the output is the probability that a particular datapoint belongs to **class 1**. What will the logistic regression method predict in our formulation?

#### Model Fitting

In [ ]:
from sklearn.linear_model import LogisticRegression

# We will now create an instance of a Logistic Regression model and fit it on our training data.
# Parameters are constantly updated as the model is being trained. We stop training our model when the parameters converge.
# We can set the number of maximum iterations that we want our model to go through in order to fit the data.

log_reg_model = LogisticRegression(max_iter=500).fit(X_train, y_train)

Our model has now been fit! We will see how well the model does on the test data. First, let's make predictions on our test data.

In [ ]:
print(log_reg_model.predict(X_test))

# But, we said that the output of logistic regression is a probability.
# What is happening? Can we look at the probabilities?

print(log_reg_model.predict_proba(X_test))

# Logistic regression has a threshold of .5.
# If the probability of predicting 1 > 0.5 (the second column in the output), the output is 1
# Otherwise, the output is 0.

####Evaluating Model Performance

The simplest metric to predict model performance is accuracy. Do you remember the definition of accuracy?

In [ ]:
log_reg_model.score(X_test, y_test)

In [ ]:
log_reg_model.score(X_train, y_train)

Not all errors are equal.
If a patient has a benign tumor and we predict it as malignant, is that more harmful than if a patient has a malignant tumor which we predict as benign?

True positive, false positive, true negative and false negative capture the different ways a model is right or wrong. A confusion matrix displays these effectively.

An easy way to remember definitions:
The positive or negative refer to model predictions of whether something has a positive class label or a negative class label. True or false refers to the truth/target values.

In [ ]:
#clinical classification not all classification errors are equal
y_pred = log_reg_model.predict(X_test)
log_reg_confusion_matrix = sklearn.metrics.confusion_matrix(y_test, y_pred)
log_reg_cm_display = sklearn.metrics.ConfusionMatrixDisplay(log_reg_confusion_matrix)
log_reg_cm_display.plot()

#### Recap

The structure of building a ML model using sklearn is as follows.
* Create an instance of a model
* Fit the model on your training data
* Evaluate your model on the test data appropriately

You can combine the creation of a model class instance and the fitting in your code.

### Decision Trees

Decision trees are a particularly popular classification method because they are interpretable. Essentially, you can think of a decision tree as someone asking you a set of questions based on the input features and then telling you whether your input is from the '0' class or the '1' class. We will visualize decision trees and that will make all of this clearer! Decision trees can be used both for classification and regression. Since we're using sklearn for decision trees, the code you see will be very similar to the code for logistic regression (and this makes coding basic models in sklearn very convenient!)

In [ ]:
# Importing the DecisionTreeClassifier class, creating an instance and fitting training data
from sklearn.tree import DecisionTreeClassifier
dec_tree_model = DecisionTreeClassifier(max_depth=2).fit(X_train, y_train)

In [ ]:
# Visualizing the decision tree model
from sklearn import tree
tree.plot_tree(dec_tree_model)

In [ ]:
# Computing the accuracy of the decision tree model on our test set
dec_tree_model.score(X_test, y_test)

**Hyperparameter:** The max depth in our case is an example of a hyperparameter. It's something that the user sets or makes a decision about. It's not a parameter that the algorithm learns. Hyperparameter tuning is an integral part of machine learning and we will only be taking a look at it very briefly!

### **Exercise 1: Random Forests**
Can you now look at the [documentation](https://scikit-learn.org/1.5/modules/generated/sklearn.ensemble.RandomForestClassifier.html) for random forests (which are collections of decision trees) and train a random forest for the task at hand? \\
a) How well does a random forest perform in comparison to a decision tree of the same depth? \\
b) What is the parameter 'n_estimators' used to control? Is that a hyperparameter?

In [ ]:
# TO DO
# Your code goes here

# Import RandomForestClasssifier from sklearn.ensemble
from sklearn.ensemble import RandomForestClassifier
# Create a RandomForestClassifier with max_depth 2
# Fit the RandomForestClassifier that you defined on your training data
ran_for_cls = RandomForestClassifier(max_depth=2).fit(X_train, y_train)
# Score the RandomForestClassifier on your test data!
ran_for_cls.score(X_test, y_test)

In [ ]:
# TO DO
# Your code goes here

# Can you create a RandomForestClassifier with a different max_depth and
# a value for n_estimators that is different from the default value?
# Can you create a RandomForestClassifier with a different max_depth and
RanFor_3_50 = sklearn.ensemble.RandomForestClassifier(max_depth=3, n_estimators=50).fit(X_train, y_train)
print("RanFor_3_50:", RanFor_3_50.score(X_test, y_test))
RanFor_3_100 = sklearn.ensemble.RandomForestClassifier(max_depth=3, n_estimators=100).fit(X_train, y_train)
print("RanFor_3_100:", RanFor_3_100.score(X_test, y_test))
RanFor_2_100 = sklearn.ensemble.RandomForestClassifier(max_depth=2, n_estimators=100).fit(X_train, y_train)
print("RanFor_2_100:", RanFor_2_100.score(X_test, y_test))
# Train it and see how it does on your test set


##Modern Approaches to ML

### Neural Networks

Neural networks are an incredibly powerful method to capture complex patterns in data and make predictions. You can look up the universal approximation theorem to understand what makes neural networks so powerful. Today, we will look at a simple neural network and how to train it.

We have an input layer with 3 nodes corresponding to 3 features. In our case, we'd have 21 nodes in our input layer.

node --> math neuron --> product

There is a hidden layer with 2 nodes and an output layer with 2 nodes. In our example, we might have an output layer with 2 nodes corresponding to the probabilities that a particular tumor is benign and that it is malignant. It could also be a single value that corresponds to the probability that the tumor is benign (or) a single value that corresponds to the probability that the tumor is malignant.

#### Creating a neural network

We will now try coding a simple neural network using sklearn again! \\
What we will experiment with, is the number of hidden layers and the number of nodes in each layer. \\
MLPClassifier can take an input hidden_layer_sizes. We can fill it in with as many values as we'd like hidden layers. The values that we fill in will be the number of nodes in each hidden layer.

In [ ]:
from sklearn.neural_network import MLPClassifier
# Let's now create a neural network with 2 hidden layers.
# The first hidden layer will have 10 nodes and the second will have 5.
# We will train it on the training set and evaluate it on the test set
nn_10_5 = MLPClassifier(alpha=1e-05, max_iter=1000, hidden_layer_sizes=(10, 5), random_state=1)
nn_10_5.fit(X_train, y_train)
nn_10_5_score = nn_10_5.score(X_test, y_test)
print(nn_10_5_score)

In [ ]:
# Plotting the training and validation loss
plt.plot(nn_10_5.loss_curve_)
# Does this make sense?

In [ ]:
# TO DO
# Can you try creating a neural network with 1 hidden layer?
# Set the hidden layer to have 4 nodes
# How does the accuracy of this neural network compare to the accuracy of the previous one?
# Your code goes here
nn_10_5 = MLPClassifier(alpha=1e-05, max_iter=1000, hidden_layer_sizes=(1, 4), random_state=1)
nn_10_5.fit(X_train, y_train)
nn_10_5_score = nn_10_5.score(X_test, y_test)
print(nn_10_5_score)



#### Learning Curves, Overfitting, Underfitting

So far, we've taken our data, split it into train and test. We've trained our model on the training data and evaluated it on the test data. Now, we will split our data into train, validate and test. We will train our model on the train data, pick hyperparameters according to model performance on validation data, and eventually, get a sense of the model's performance on the test data!

We will first plot learning curves for a neural network with 3 layers in total, and with 8 nodes in the hidden layer to understand overfitting and underfitting. We will then move on to hyperparameter tuning.

In [ ]:
# Let's take our data, X and y. Let's split the data into train_val and test.
# Let 20% of your data be in the test set and let the other 80% go into train_val.

# We will create X_train_val, X_test, y_train_val, y_test using the train_test_split function that you already imported!
X_train_val, X_test, y_train_val, y_test = train_test_split(X, y, test_size=0.2, random_state=1)

In [ ]:
# Let us now split X_train_val, y_train_val into X_train, X_val, y_train and y_val
# We will have 30% of X_train_val in X_val and the remaining in X_train.

# TO DO
# Create X_train, X_val, y_train, y_val
# Your code goes here

X_train, X_val, y_train, y_val = train_test_split(X_train_val, y_train_val, test_size=0.3, random_state = 1)


In [ ]:
# Computing train and validation losses for a neural network with 1 hidden layer with 8 nodes
# This model is trained for 4000 epochs

from sklearn.metrics import log_loss

nn = MLPClassifier(alpha=1e-05, max_iter=1, hidden_layer_sizes=(8,10), random_state=1)
val_loss_array = []
train_loss_array = []
for epoch in range(4000):
  # Train the neural network on the train set
  nn.partial_fit(X_train, y_train, np.unique(y_train))

  # Evaluate the neural network on the validation set
  train_loss_array.append(nn.loss_)
  y_val_pred_proba = nn.predict_proba(X_val)
  val_loss = log_loss(y_val, y_val_pred_proba)
  val_loss_array.append(val_loss)

In [ ]:
# Plot the validation and training scores
plt.plot(val_loss_array, label = 'validation')
plt.plot(train_loss_array, label = 'train')
plt.xlabel('Epoch number')
plt.ylabel('Loss')
plt.legend()
plt.show()

It is important to plot training and validation losses to see when you should stop training. We run the risk of overfitting if we train models beyond a point. We will see the validation loss increase while the training loss reduces. You can look at this [link](https://machinelearningmastery.com/learning-curves-for-diagnosing-machine-learning-model-performance/) to learn more about overfitting, underfitting etc.

#### Hyperparameter Tuning

Hyperparameter tuning is crucial to training models to optimize performance. Let's take a look at tuning the number of nodes in the hidden layer of a neural network with 3 layers in total

In [ ]:
# We will now do some hyperparameter tuning!
# Let's create a neural network with 1 hidden layer.
# We will decide what the size of the hidden layer should be from a set of options - 4, 8, 16

for i in [4, 8, 16]:
  # Create a neural network
  nn = MLPClassifier(alpha=1e-05, max_iter=2000, hidden_layer_sizes=(i,), random_state=1)

  # Train the neural network on the train set
  nn.fit(X_train, y_train)

  # Evaluate the neural network on the validation set
  val_score = nn.score(X_val, y_val)

  # Print the validation score
  print(f'Validation score for {i} nodes: {val_score}')

In [ ]:
# TO DO
# Pick the number of nodes 'n' in the hidden layer based on validation error
# Let us now see how well a neural network with 'n' nodes in the hidden layer does on the test set.
nn_8 = MLPClassifier(alpha=1e-05, max_iter=4000, hidden_layer_sizes=(8,), random_state=1)
# First, let's set up our neural network and train it on the train set
# Your code goes here
nn_8.fit(X_train, y_train)

# Now, let us evaluate it on the test set
# Your code goes here
print(nn.score(X_test, y_test))

#### k-fold cross-validation

Can you think of any problems that arise with how we did things using a single split of train-validation-test set? \\
The test error is dependent on how we partitioned our data into train and validation. If we want a more robust estimate of how well our models perform, we would do k-fold cross validation.

Here's how k-fold cross-validation is performed.
* We split our data into training and test sets.
* We take our training data and split it into k-folds of roughly equal size.
* We train on k-1 folds and validate our model on the remaining fold, repeating this procedure iteratively. In each iteration, one of the k-folds will act as the validation set with the other k-1 being used to train the model.
* At the end of the procedure, we average the validation scores from the k runs to choose our model and/or report a validation score.

The figure below will illustrate this!

In [ ]:
# We will now use k-fold crossvalidation with k = 5 to get an estimate of how well our model is doing
# For this example, let us create a neural network with 4 nodes in the hidden layer
from sklearn.model_selection import cross_val_score
nn = MLPClassifier(hidden_layer_sizes = (4, ), max_iter = 5000)
scores = cross_val_score(nn, X_train_val, y_train_val, cv=5)

In [ ]:
# scores is a list with the 5 validation scores
print(scores)
# We will now print the mean and standard deviation of the scores
print(scores.mean(), scores.std())

In [ ]:
# Let's evaluate the performance of the neural network on the test set again
nn.fit(X_train_val, y_train_val)
print(nn.score(X_test, y_test))

In [ ]:
# We will now visualize one of these datapoints
import matplotlib.pyplot as plt
plt.imshow(digits.images[100], cmap = 'gray')
print(digits.target[100])

In [ ]:
# Splitting data into train and test
from sklearn.model_selection import train_test_split
X_digits_train, X_digits_test, y_digits_train, y_digits_test = train_test_split(digits.data, digits.target, test_size=0.2, random_state=41)